In [ ]:
import logging
import ipywidgets as widgets
from IPython.display import display

# Configure a basic logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logger = logging.getLogger("demo")

# Map slider positions to logging levels
level_map = {
    0: logging.DEBUG,
    1: logging.INFO,
    2: logging.WARNING,
    3: logging.ERROR,
    4: logging.CRITICAL
}
level_names = {v: k for k, v in level_map.items()}

# Create the slider
slider = widgets.IntSlider(
    value=1, min=0, max=4, step=1,
    description=f"Logging Level: {labels[1]}",
    continuous_update=True,
    readout=False,  # Hide numeric value entirely
    layout=widgets.Layout(width='400px'),
    style = {'description_width': '150px'}
)

slider.style.handle_color = 'lightblue'

# Labels for levels
labels = ['DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL']
slider_ticks = widgets.HBox([widgets.Label(l) for l in labels])

# Callback to update logging level
def update_level(change):
    new_val = change['new']
    new_level = level_map[change['new']]
    slider.description = f"Logging Level: {labels[new_val]}"
    logger.setLevel(new_level)
    logger.log(new_level, f"Logging level set to {logging.getLevelName(new_level)}")

slider.observe(update_level, names='value')

display(slider)

# Example: function to test logging output
def test_logging():
    logger.debug("This is a DEBUG message.")
    logger.info("This is an INFO message.")
    logger.warning("This is a WARNING message.")
    logger.error("This is an ERROR message.")
    logger.critical("This is a CRITICAL message.")

# Call test_logging() after moving the slider to see filtered output

IntSlider(value=1, description='Level: INFO', layout=Layout(width='400px'), max=4, readout=False, style=Slider…

In [6]:
test_logging()

2025-09-16 12:40:49,395 [DEBUG] demo: This is a DEBUG message.
2025-09-16 12:40:49,395 [INFO] demo: This is an INFO message.
2025-09-16 12:40:49,395 [WARNING] demo: This is a WARNING message.
2025-09-16 12:40:49,396 [ERROR] demo: This is an ERROR message.
2025-09-16 12:40:49,396 [CRITICAL] demo: This is a CRITICAL message.


# Example: Oscillating Droplet

In this example we start with an elliptic droplet. The surface tension force deforms it as it strives to minimize the surface. In other words it will always try to form the droplet towards a cirlce. Due to the viscosity however it will slightly overcorrect such that a wobbling movement occurs. The following PDE describes this problem:

\begin{align*}
\rho \partial_t \mathbf{u} 
  - \operatorname{div}(2 \mu \varepsilon(\mathbf{u})) + \nabla p &= 0
  && \text{in } {\Omega(t)} \\
\nabla \cdot \mathbf{u} &= 0
  && \text{in } {\Omega(t)} \\
\boldsymbol{\sigma} \cdot \mathbf{n}_\Gamma &= { \tau \kappa \mathbf{n}_\Gamma}
  && \text{on } {\Gamma(t)} \\
  { \mathbf{u} \cdot \mathbf{n}_\Gamma} &{= \mathcal{V}_\Gamma} && \text{on } {\Gamma(t)}
\end{align*}

where $\tau$ is a surface tension coefficient, $\kappa$ is the mean curvature and $\mathcal{V}_\Gamma$ is the velocity of the interface in normal dircetion.

First we import the necessary modules and create the mesh

In [2]:
from ngsolve import *
from xfem import *
import ngsolve.webgui as ngw
from netgen.occ import *

from ngsxditto.transport import *
from ngsxditto.levelset import *
from ngsxditto.fluid import *
from ngsxditto.redistancing import *
from ngsxditto.gradient_tester import *
from ngsxditto.velocity_extension import *
from ngsxditto.solver import *
from ngsxditto.visualization import *

2025-09-16 12:38:32,564 [INFO] ngsxditto: importing ngsxditto
2025-09-16 12:38:32,564 [DEBUG] ngsxditto.transport: importing ngsxditto.transport
2025-09-16 12:38:32,569 [DEBUG] ngsxditto.levelset: importing ngsxditto.levelset
2025-09-16 12:38:32,570 [DEBUG] ngsxditto.redistancing: importing ngsxditto.redistancing
2025-09-16 12:38:32,632 [DEBUG] matplotlib: matplotlib data path: /usr/lib/python3.13/site-packages/matplotlib/mpl-data
2025-09-16 12:38:32,639 [DEBUG] matplotlib: CONFIGDIR=/home/schruste/.config/matplotlib
2025-09-16 12:38:32,666 [DEBUG] matplotlib: interactive is False
2025-09-16 12:38:32,666 [DEBUG] matplotlib: platform is linux
2025-09-16 12:38:32,707 [DEBUG] matplotlib: CACHEDIR=/home/schruste/.cache/matplotlib
2025-09-16 12:38:32,712 [DEBUG] matplotlib.font_manager: Using fontManager instance from /home/schruste/.cache/matplotlib/fontlist-v390.json


importing ngsxfem-2.1.2504


In [ ]:
domain = MoveTo(-1, -1).Rectangle(2, 2).Face()
domain.edges.Max(X).name = "right"
domain.edges.Min(X).name = "left"
domain.edges.Min(Y).name = "bottom"
domain.edges.Max(Y).name = "top"
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=0.15))

We define our initial geometry.

In [ ]:
dt = 0.02
t = Parameter(0)
starting_levelset = (5*x**2 + y**2)**(1/2) - 2.0/3.0
transport = ExplicitDGTransport(mesh, dt=dt, order=2, compile=False)
levelset = LevelSetGeometry(transport)
levelset.Initialize(starting_levelset)
ngw.Draw(levelset.field)

We use Taylor-Hood elements. As a simplification we only consider the (unsteady) Stokes problem. We calculate the curvature of the level set geometry and add the resulting tension force to the right hand side in our Stokes solver. We assume we have no initial velocity.

In [ ]:
fluid_params = FluidParameters(viscosity=2e-2)

mean_curvature = MeanCurvatureSolver(mesh, order=2, lset=levelset)
mean_curvature.Compute()

fluid = TaylorHood(mesh, fluid_params, lset=levelset, f=CF((0, 0)), surface_tension=mean_curvature.H, dt=dt, order=3, ghost_stab=1)
fluid.Initialize(initial_velocity=CF((0, 0)))

For the unsteady Stokes problem we now want to update our level set based on this field, i.e. $$\mathbf{u} \cdot \mathbf{n}_\Gamma = \mathcal{V}_\Gamma$$ where $\mathcal{V}_\Gamma$ is the velocity of the interface in normal direction. For our level set update we need a velocity field $w$ on the whole domain, not just on the interface. For this we extend the velocity field using a diffusion based algorithm. After our level set update we can then calculate the curvature again to solve a time-step of the Stokes problem.

In [ ]:
velocity_extension = DiffusionBasedVelocityExtension(levelset)

end_time = 8

time_loop = TimeLoop(time=t, dt=dt, end_time=end_time)

sphericity = SphericityDiagram(levelset, time=t, name="sphericity")
animation = VelocityAnimation(levelset, fluid, time=t, end_time=end_time, name="animation",
                              min=-0.075, max=0.225, autoscale=False)

time_loop.AddVisualization(sphericity)
time_loop.AddVisualization(animation)

time_loop.Register(velocity_extension.SolveVelocity, fluid.gfu.components[0], name="vel ext.")
time_loop.Register(levelset.OneStep, name="levelset")
time_loop.Register(mean_curvature.Compute, name="mean curvature")
time_loop.Register(fluid.OneStep, name="moving stokes")

time_loop()

The sphericity diagram shows the ratio of the surface area (perimeter) and the volume (area) of a shape. We observe that starting from an initial deformation this ratio has periodic declining behavior.